In [ ]:
"""
code to extract hypoxia metrics from model before and after bias-correction
so it can be used to compare difference between bias-corrected and to be plotted by another notebook
"""

In [29]:
from scipy.stats import ttest_ind
import xarray as xr
import numpy as np
import pandas as pd
from datetime import datetime
from lo_tools import Lfun
from lo_tools import plotting_functions as pfun

In [30]:
Ldir = Lfun.Lstart()
gridname = 'cas2k'
pth = Ldir['LOo'] / 'pgrid' /gridname
in_fn = pth/'grid.nc'
# load the data
grid = xr.open_dataset(in_fn)

In [31]:
gtx = 'cas2k_v4_lo2b'
fod1 = 'bias_corrected'
in_dir0 = Ldir['LOo'] / 'extract'/ gtx / 'box' / fod1

In [32]:
# turn on and off based on different trails
add_hake_hpic_trawl = True
cast_split = True
cast_balanced = True
use_z_fraction = True
benthic_aware = False
hypoxia_aware = False
hypoxia_shelf_aware = False
hypoxia_shelf_smooth_aware = False
hypoxia_two_tier_shelf_aware = True
remove_NO3 = True

if add_hake_hpic_trawl:
    runame = 'with_hake_hpic_trawl'
    if cast_split:
        name = 'cid_split'
        if cast_balanced:
            name += '_cast_balanced'
            if cast_balanced and use_z_fraction:
                name += '_z_fraction'
            if cast_balanced and benthic_aware:
                name += '_benthic_aware'
            if cast_balanced and hypoxia_aware:
                name += '_hypoxia_aware'
            if cast_balanced and hypoxia_shelf_aware:
                name += '_hypoxia_shelf_aware'
            if cast_balanced and hypoxia_shelf_smooth_aware:
                name += '_hypoxia_shelf_smooth_aware'
            if cast_balanced and hypoxia_two_tier_shelf_aware:
                name += '_hypoxia_two_tier_shelf_aware'
            if remove_NO3:
                name += '_no_NO3'

    else:
        name = ''
        
in_dir = in_dir0/ runame / name
out_dir = in_dir / 'hypoxia'
fig_dir = in_dir / 'figures'
Lfun.make_dir(out_dir)
Lfun.make_dir(fig_dir)

In [34]:
testing = False
if testing:
    year_list = [1993]
    # year_list = np.arange(1998,2001)
else:
    year_list = np.arange(1993,2023)

In [35]:
filter = True
for year in year_list:
    print(year)
    out_fn = out_dir / f'hypoxia_daily_bot_{year}.nc'
    fn = str(in_dir/f'daily_bot_{year}_CO2SYS.nc')

    ds = xr.open_dataset(fn)
    ds = ds.sel(ocean_time=ds.ocean_time.dt.month.isin(range(4,10)))

    lon = ds.lon_rho.values
    lat = ds.lat_rho.values


    pm_trim = grid.pm.sel(eta_rho=ds.eta_rho, xi_rho=ds.xi_rho)
    pn_trim = grid.pn.sel(eta_rho=ds.eta_rho, xi_rho=ds.xi_rho)

    DX = 1 / pm_trim.values
    DY = 1 / pn_trim.values
    DA = DX * DY   # horizontal cell area [m2]
    # -------------------------------------------------
    # Time
    # -------------------------------------------------
    ot = ds.ocean_time.data
    oxy = ds.oxygen.values

    valid = np.isfinite(oxy)
    hyp_mask = (oxy <= 62.2) & valid

    # Per-grid hypoxia metrics
    # hyp_days, hyp_frac, hyp_area_total shape: (eta_rho, xi_rho)
    # -------------------------------------------------
    hyp_days = np.sum(hyp_mask, axis=0)

    # hyp_frac, 0-1
    valid_count = np.sum(valid, axis=0)

    hyp_frac = np.divide(
        hyp_days,
        valid_count,
        out=np.full(hyp_days.shape, np.nan, dtype=float),
        where=valid_count > 0
    )

    # cumulative hypoxic area exposure
    hyp_area_total = DA * hyp_days
    # -------------------------------------------------
    # Daily area [m2]
    # hyp_area shape: (ocean_time,) only keep OR-WA
    hyp_mask0 = np.where(
        (oxy <= 62.2) &
        np.isfinite(oxy) &
        (lat <= 48.3),
        1.0,
        np.nan
    )

    hyp_area = np.nansum(DA * hyp_mask0, axis=(1, 2))

    # =========================
    # corrected_DO
    # =========================
    oxy_corr = ds['corrected_DO'].values

    valid_corr = np.isfinite(oxy_corr)
    hyp_mask_corr = (oxy_corr <= 62.2) & valid_corr

    hyp_days_corr = np.sum(hyp_mask_corr, axis=0)

    valid_count_corr = np.sum(valid_corr, axis=0)
    hyp_frac_corr = np.divide(
        hyp_days_corr,
        valid_count_corr,
        out=np.full(hyp_days_corr.shape, np.nan, dtype=float),
        where=valid_count_corr > 0
    )

    hyp_area_total_corr = DA * hyp_days_corr

    hyp_mask_corr0 = np.where(
        (oxy_corr <= 62.2) &
        valid_corr &
        (lat <= 48.3),
        1.0,
        np.nan
    )

    hyp_area_corr = np.nansum(DA * hyp_mask_corr0, axis=(1, 2))

    # -------------------------------------------------
    # Put into output dataset
    # -------------------------------------------------
    ds1 = xr.Dataset()

    ds1['ocean_time'] = ('ocean_time', ot)
    ds1['ocean_time'].attrs = ds['ocean_time'].attrs

    ds1['hyp_area'] = (
        ('ocean_time',),
        hyp_area,
        {'units': 'm^2',
         'long_name': 'Total shelf bottom area with oxygen <= 62.2 mmol m-3'}
    )

    ds1['hyp_days'] = (
        ('year','eta_rho', 'xi_rho'),
        hyp_days[None, :, :].astype(np.int32),
        {
            'units': 'count',
            'long_name': 'Number of days with bottom oxygen <= 62.2 mmol m-3'
        }
    )
    ds1['year'] = ('year', [year])

    ds1['hyp_frac'] = (
        ('year','eta_rho', 'xi_rho'),
        hyp_frac[None, :, :],
        {'units': '1',
         'long_name': 'Fraction of time with bottom oxygen <= 62.2 mmol m-3'}
    )

    ds1['hyp_area_total'] = (
        ('year','eta_rho', 'xi_rho'),
        hyp_area_total[None, :, :],
        {'units': 'm^2 day',
         'long_name': 'Cumulative hypoxic area exposure (area x days)'}
    )
    # -------------------------
    # corrected_DO outputs
    # -------------------------
    ds1['hyp_area_corrected'] = (
        ('ocean_time',),
        hyp_area_corr,
        {'units': 'm^2',
         'long_name': 'Total shelf bottom area with corrected_DO <= 62.2 mmol m-3'}
    )

    ds1['hyp_days_corrected'] = (
        ('year','eta_rho', 'xi_rho'),
        hyp_days_corr[None, :, :].astype(np.int32),
        {
        'units': 'count',
         'long_name': 'Number of days with corrected_DO <= 62.2 mmol m-3'}
    )

    ds1['hyp_frac_corrected'] = (
        ('year','eta_rho', 'xi_rho'),
        hyp_frac_corr[None, :, :],
        {'units': '1',
         'long_name': 'Fraction of time with corrected_DO <= 62.2 mmol m-3'}
    )

    ds1['hyp_area_total_corrected'] = (
        ('year','eta_rho', 'xi_rho'),
        hyp_area_total_corr[None, :, :],
        {'units': 'm^2 day',
         'long_name': 'Cumulative hypoxic area exposure (corrected_DO)'}
    )
    ds1['Lat'] = (
        ('eta_rho', 'xi_rho'),
        lat,
        {'units': 'degree_north',
         'long_name': 'Latitude of RHO-points'}
    )

    ds1['Lon'] = (
        ('eta_rho', 'xi_rho'),
        lon,
        {'units': 'degree_east',
         'long_name': 'Longitude of RHO-points'}
    )

    ds1['DA'] = (
        ('eta_rho', 'xi_rho'),
        DA,
        {'units': 'm^2',
         'long_name': 'Cell horizontal area'}
    )

    # metadata
    ds1.attrs['hypoxia_threshold'] = '62.2 mmol m-3'
    ds1.attrs['description'] = 'Bottom hypoxia metrics for oxygen and corrected_DO'

    # Save
    # -------------------------------------------------
    ds1.to_netcdf(out_fn, unlimited_dims='ocean_time')
    ds.close()
    ds1.close()

1993
1994
1995
1996
1997
1998
1999
2000
2001
2002
2003
2004
2005
2006
2007
2008
2009
2010
2011
2012
2013
2014
2015
2016
2017
2018
2019
2020
2021
2022
